# Pulse simulation calibration

This notebook mirrors the local QUBEX simulator examples before using any provider-generated schedule:

- `/home/orangekame3/src/github.com/orangekame3/qubex/docs/examples/simulator/6_drag_calibration.ipynb`
- `/home/orangekame3/src/github.com/orangekame3/qubex/docs/examples/simulator/8_cr_calibration.ipynb`

It defines the same standalone simulator model, builds a DRAG half-pi pulse and a calibrated echoed ZX90 pulse, then checks both with repeated sequences. This is the baseline we need before attributing population-dynamics errors to scheduling.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import qutip as qt

import qubex as qx
from qxsimulator import Control, Coupling, QuantumSimulator, QuantumSystem, SimulationResult, Transmon
from qubex.measurement.classifiers.state_classifier_gmm import StateClassifierGMM

qx.pulse.set_sampling_period(0.1)

# Units: GHz and ns, matching the QUBEX simulator docs.
TRANSMON_DIMENSION = 3

CONTROL_QUBIT_LABEL = "Q04"
CONTROL_QUBIT_FREQUENCY = 7157.231e-3
CONTROL_ANHARMONICITY = -393.715e-3

TARGET_QUBIT_LABEL = "Q01"
TARGET_QUBIT_FREQUENCY = 8032.295e-3
TARGET_ANHARMONICITY = -487.412e-3

RELAXATION_RATE = 0.05e-3
DEPHASING_RATE = 0.05e-3
COUPLING_STRENGTH = 5e-3
FREQUENCY_DIFF = CONTROL_QUBIT_FREQUENCY - TARGET_QUBIT_FREQUENCY

DRAG_DURATION = 24
CR_AMPLITUDE_MAX = 0.75 * abs(FREQUENCY_DIFF)
CR_PHASE_OFFSET = 0.25 * np.pi
CR_RAMPTIME = 16
XT_RATIO = 0.01
XT_PHASE_OFFSET = 0.5 * np.pi
ROTARY_MULTIPLE = 17
DURATION_UNIT = 16

# QUBEX docs use 1000 ns for CR Hamiltonian tomography. Keep the
# default lighter for notebook iteration; set this to 1000 for a final run.
TOMOGRAPHY_DURATION = 400
TOMOGRAPHY_SAMPLES = 80


## Define the simulator model

This is intentionally the same kind of model as the QUBEX CR calibration notebook: two coupled three-level transmons with fixed frequencies, anharmonicities, coherence rates, and coupling.


In [ ]:
control_qubit = Transmon(
    label=CONTROL_QUBIT_LABEL,
    dimension=TRANSMON_DIMENSION,
    frequency=CONTROL_QUBIT_FREQUENCY,
    anharmonicity=CONTROL_ANHARMONICITY,
    relaxation_rate=RELAXATION_RATE,
    dephasing_rate=DEPHASING_RATE,
)

target_qubit = Transmon(
    label=TARGET_QUBIT_LABEL,
    dimension=TRANSMON_DIMENSION,
    frequency=TARGET_QUBIT_FREQUENCY,
    anharmonicity=TARGET_ANHARMONICITY,
    relaxation_rate=RELAXATION_RATE,
    dephasing_rate=DEPHASING_RATE,
)

system = QuantumSystem(
    objects=[control_qubit, target_qubit],
    couplings=[Coupling(pair=(CONTROL_QUBIT_LABEL, TARGET_QUBIT_LABEL), strength=COUPLING_STRENGTH)],
)

simulator = QuantumSimulator(system)


## Build and check a DRAG half-pi pulse

The QUBEX experiment docs use `exp.repeat_sequence({Q0: hpi_pulse}, repetitions=20)` after calibration. Offline we mirror that behavior by sweeping repetition count `N = 0..repetitions` and simulating the repeated waveform for each `N`.

For a good HPI pulse, the sweep should accumulate as expected: `N=1` near a half rotation, `N=2` near a pi pulse, and `N=4` near identity.


In [ ]:
def drag_pulse(duration: float, amplitude: float, beta: float, *, pulse_type: str = "Gaussian"):
    return qx.pulse.Drag(duration=duration, amplitude=amplitude, beta=beta, type=pulse_type)


def area_normalized_drag(duration: float, area: float, beta: float):
    pulse = drag_pulse(duration=duration, amplitude=1.0, beta=beta)
    norm_factor = area / float(np.sum(np.abs(pulse.values) * pulse.SAMPLING_PERIOD))
    return pulse.scaled(norm_factor)


hpi_beta = -0.5 / (2 * np.pi * control_qubit.anharmonicity)
hpi_pulse = area_normalized_drag(DRAG_DURATION, np.pi / 2, hpi_beta)
pi_pulse = area_normalized_drag(DRAG_DURATION, np.pi, hpi_beta)

print("HPI beta:", hpi_beta)
print("HPI max amplitude:", float(np.max(hpi_pulse.real)))
print("PI max amplitude:", float(np.max(pi_pulse.real)))
hpi_pulse.plot(title="DRAG HPI pulse", divide_by_two_pi=True)


In [ ]:
def simulate_control_pulse(waveform, *, n_samples: int = 2) -> SimulationResult:
    return simulator.mesolve(
        controls=[Control(target=CONTROL_QUBIT_LABEL, frequency=CONTROL_QUBIT_FREQUENCY, waveform=waveform)],
        initial_state={CONTROL_QUBIT_LABEL: "0", TARGET_QUBIT_LABEL: "0"},
        n_samples=n_samples,
    )


def single_qubit_populations(result: SimulationResult) -> dict[str, float]:
    populations = result._get_population(result.get_substates(CONTROL_QUBIT_LABEL)[-1])
    return {str(index): float(value) for index, value in enumerate(np.real(populations))}


def repeat_hpi_sequence(pulse, *, repetitions: int = 20) -> pd.DataFrame:
    rows = []
    for repeats in range(repetitions + 1):
        if repeats == 0:
            rows.append({"repeats": 0, "0": 1.0, "1": 0.0, "2": 0.0, "bloch": [0.0, 0.0, 1.0]})
            continue
        waveform = qx.PulseArray([pulse] * repeats)
        result = simulate_control_pulse(waveform)
        row = {"repeats": repeats, **single_qubit_populations(result)}
        row["bloch"] = np.round(result.get_bloch_vectors(CONTROL_QUBIT_LABEL)[-1], 4).tolist()
        rows.append(row)
    return pd.DataFrame(rows)


hpi_repeat_df = repeat_hpi_sequence(hpi_pulse, repetitions=20)
display(hpi_repeat_df)

fig = qx.viz.make_plot_figure(
    x=hpi_repeat_df["repeats"],
    y=hpi_repeat_df["1"],
    title=f"Repeat sequence : {CONTROL_QUBIT_LABEL}",
    xlabel="Number of repetitions",
    ylabel="Normalized signal",
    ylim=[0, 1],
)
fig.show()


## Build CR and echoed ZX90 sequences

This follows the QUBEX CR calibration notebook. First estimate the effective Hamiltonian terms from target-qubit Bloch motion, then choose CR phase, cancellation tone, ZX90 duration, and echoed CR amplitude. The docs use a long tomography window; this notebook exposes `TOMOGRAPHY_DURATION` so iteration can stay light.


In [ ]:
def cr_sequence(
    *,
    duration: float,
    cr_amplitude: float,
    cr_phase: float,
    cancel_amplitude: float,
    cancel_phase: float,
    echo: bool,
):
    cr_beta = -1 / (2 * np.pi * FREQUENCY_DIFF)
    cr_waveform = qx.pulse.FlatTop(
        duration=duration,
        amplitude=2 * np.pi * cr_amplitude,
        tau=CR_RAMPTIME,
        phase=cr_phase + CR_PHASE_OFFSET,
        beta=cr_beta,
    )
    cancel_waveform = qx.pulse.FlatTop(
        duration=duration,
        amplitude=2 * np.pi * cancel_amplitude,
        tau=CR_RAMPTIME,
        phase=cancel_phase,
        beta=-1 / (2 * np.pi * target_qubit.anharmonicity),
    )
    crosstalk_waveform = qx.pulse.FlatTop(
        duration=duration,
        amplitude=2 * np.pi * cr_amplitude * XT_RATIO,
        tau=CR_RAMPTIME,
        phase=XT_PHASE_OFFSET,
        beta=cr_beta,
    )
    channels = [
        qx.PulseChannel(label="Control", frequency=CONTROL_QUBIT_FREQUENCY, target=CONTROL_QUBIT_LABEL),
        qx.PulseChannel(label="CR", frequency=TARGET_QUBIT_FREQUENCY, target=CONTROL_QUBIT_LABEL),
        qx.PulseChannel(label="Crosstalk", frequency=TARGET_QUBIT_FREQUENCY, target=TARGET_QUBIT_LABEL),
        qx.PulseChannel(label="Target", frequency=TARGET_QUBIT_FREQUENCY, target=TARGET_QUBIT_LABEL),
    ]
    with qx.PulseSchedule(channels) as cr:
        cr.add("CR", cr_waveform)
        cr.add("Crosstalk", crosstalk_waveform)
        cr.add("Target", cancel_waveform)

    if not echo:
        return cr

    with qx.PulseSchedule(channels) as ecr:
        ecr.call(cr)
        ecr.barrier()
        ecr.add("Control", pi_pulse)
        ecr.barrier()
        ecr.call(cr.scaled(-1))
        ecr.barrier()
        ecr.add("Control", pi_pulse)
    return ecr


def simulate_cr(
    *,
    cr_amplitude: float,
    cr_phase: float,
    cancel_amplitude: float,
    cancel_phase: float,
    duration: float,
    echo: bool,
    control_state: str,
    n_samples: int = 120,
) -> SimulationResult:
    return simulator.mesolve(
        controls=cr_sequence(
            duration=duration,
            cr_amplitude=cr_amplitude,
            cr_phase=cr_phase,
            cancel_amplitude=cancel_amplitude,
            cancel_phase=cancel_phase,
            echo=echo,
        ),
        initial_state={CONTROL_QUBIT_LABEL: control_state, TARGET_QUBIT_LABEL: "0"},
        n_samples=n_samples,
    )


In [ ]:
def hamiltonian_tomography(
    *,
    cr_amplitude: float,
    cr_phase: float,
    cancel_amplitude: float,
    cancel_phase: float,
    duration: float = TOMOGRAPHY_DURATION,
    n_samples: int = TOMOGRAPHY_SAMPLES,
) -> dict:
    result_0 = simulate_cr(
        cr_amplitude=cr_amplitude,
        cr_phase=cr_phase,
        cancel_amplitude=cancel_amplitude,
        cancel_phase=cancel_phase,
        duration=duration,
        echo=False,
        control_state="0",
        n_samples=n_samples,
    )
    result_1 = simulate_cr(
        cr_amplitude=cr_amplitude,
        cr_phase=cr_phase,
        cancel_amplitude=cancel_amplitude,
        cancel_phase=cancel_phase,
        duration=duration,
        echo=False,
        control_state="1",
        n_samples=n_samples,
    )

    times = result_0.times
    vectors_0 = result_0.get_bloch_vectors(TARGET_QUBIT_LABEL)
    vectors_1 = result_1.get_bloch_vectors(TARGET_QUBIT_LABEL)
    indices = (times >= CR_RAMPTIME) & (times < times[-1] - CR_RAMPTIME)
    times_fit = times[indices] - CR_RAMPTIME * 0.5

    fit_0 = qx.fit.fit_rotation(times_fit, vectors_0[indices], plot=False)
    fit_1 = qx.fit.fit_rotation(times_fit, vectors_1[indices], plot=False)
    omega = np.concatenate([0.5 * (fit_0["Omega"] + fit_1["Omega"]), 0.5 * (fit_0["Omega"] - fit_1["Omega"])])
    coeffs = dict(zip(["IX", "IY", "IZ", "ZX", "ZY", "ZZ"], omega / (2 * np.pi), strict=True))
    xt_rotation = coeffs["IX"] + 1j * coeffs["IY"]
    cr_rotation = coeffs["ZX"] + 1j * coeffs["ZY"]
    zx90_duration = 1 / (4 * np.abs(cr_rotation))

    print("Rotation coefficients [MHz]:", {key: round(value * 1e3, 3) for key, value in coeffs.items()})
    print("XT rate/phase:", float(np.abs(xt_rotation) * 1e3), float(np.angle(xt_rotation)))
    print("CR rate/phase:", float(np.abs(cr_rotation) * 1e3), float(np.angle(cr_rotation)))
    print("ZX90 duration [ns]:", float(zx90_duration))
    return {"coeffs": coeffs, "xt_rotation": xt_rotation, "cr_rotation": cr_rotation, "zx90_duration": float(zx90_duration)}


tomography_1 = hamiltonian_tomography(
    cr_amplitude=CR_AMPLITUDE_MAX,
    cr_phase=0.0,
    cancel_amplitude=0.0,
    cancel_phase=0.0,
)
cr_phase_calib = -float(np.angle(tomography_1["cr_rotation"]))
print("CR phase calibration:", cr_phase_calib)


In [ ]:
tomography_2 = hamiltonian_tomography(
    cr_amplitude=CR_AMPLITUDE_MAX,
    cr_phase=cr_phase_calib,
    cancel_amplitude=0.0,
    cancel_phase=0.0,
)

cancel_pulse_max = -tomography_2["xt_rotation"]
cancel_amplitude_max = float(np.abs(cancel_pulse_max))
cancel_phase_max = float(np.angle(cancel_pulse_max))
print("cancel amplitude/phase:", cancel_amplitude_max, cancel_phase_max)

tomography_3 = hamiltonian_tomography(
    cr_amplitude=CR_AMPLITUDE_MAX,
    cr_phase=cr_phase_calib,
    cancel_amplitude=cancel_amplitude_max,
    cancel_phase=cancel_phase_max,
)
zx_rate = float(tomography_3["coeffs"]["ZX"])
zx90_duration = tomography_3["zx90_duration"]
cr_duration = float(((zx90_duration / 2 + CR_RAMPTIME) // DURATION_UNIT + 1) * DURATION_UNIT)
print("echoed ZX90 CR duration:", cr_duration)


In [ ]:
def measure_target_z(cr_amplitude: float) -> float:
    ratio = cr_amplitude / CR_AMPLITUDE_MAX
    cancel_pulse = (cancel_pulse_max + zx_rate * ROTARY_MULTIPLE) * ratio
    result = simulate_cr(
        cr_amplitude=cr_amplitude,
        cr_phase=cr_phase_calib,
        cancel_amplitude=float(np.abs(cancel_pulse)),
        cancel_phase=float(np.angle(cancel_pulse)),
        duration=cr_duration,
        echo=True,
        control_state="0",
        n_samples=2,
    )
    return float(result.get_bloch_vectors(TARGET_QUBIT_LABEL)[-1][2])


amplitude_range = np.linspace(CR_AMPLITUDE_MAX * 0.8, CR_AMPLITUDE_MAX * 1.2, 20)
z_values = np.array([measure_target_z(float(amplitude)) for amplitude in amplitude_range])
fig = qx.viz.make_plot_figure(
    x=amplitude_range,
    y=z_values,
    title="Echoed ZX90 amplitude calibration",
    xlabel="CR amplitude [GHz]",
    ylabel="target <Z>",
    ylim=[-1.1, 1.1],
)
fig.add_hline(y=0.0, line_dash="dot", line_color="#64748b")
fig.show()

fit_result = qx.fit.fit_polynomial(x=amplitude_range, y=z_values, degree=3, plot=False)
cr_amplitude_calib = float(fit_result["root"])
ratio = cr_amplitude_calib / CR_AMPLITUDE_MAX
cancel_pulse_calib = (cancel_pulse_max + zx_rate * ROTARY_MULTIPLE) * ratio
cancel_amplitude_calib = float(np.abs(cancel_pulse_calib))
cancel_phase_calib = float(np.angle(cancel_pulse_calib))

print("CR amplitude calibration:", cr_amplitude_calib)
print("cancel amplitude/phase calibration:", cancel_amplitude_calib, cancel_phase_calib)
print("target <Z> at calibrated amplitude:", measure_target_z(cr_amplitude_calib))


## Check the ZX90 pulse with experiment-style repeats

The QUBEX experiment docs use `exp.repeat_sequence(zx90)` for CR validation. That workflow sweeps repetition count and inspects the measured response, without process fidelity.

Offline we mirror that behavior with `N = 0..repetitions`, using `zx90_sequence.repeated(N)` for each nonzero repetition count. To keep the check focused and lightweight, we use one control-qubit initial state and plot the target-qubit population.


In [ ]:
zx90_sequence = cr_sequence(
    duration=cr_duration,
    cr_amplitude=cr_amplitude_calib,
    cr_phase=cr_phase_calib,
    cancel_amplitude=cancel_amplitude_calib,
    cancel_phase=cancel_phase_calib,
    echo=True,
)

print("ZX90 pulse parameters:")
print(f"  duration:         {cr_duration} ns")
print(f"  ramptime:         {CR_RAMPTIME} ns")
print(f"  cr_amplitude:     {cr_amplitude_calib} GHz")
print(f"  cr_phase:         {cr_phase_calib} rad")
print(f"  cancel_amplitude: {cancel_amplitude_calib} GHz")
print(f"  cancel_phase:     {cancel_phase_calib} rad")
zx90_sequence.plot(title="Calibrated echoed ZX90 sequence", width=800, divide_by_two_pi=True, n_samples=500)


In [ ]:
def target_qubit_population(result: SimulationResult) -> dict[str, float]:
    populations = result._get_population(result.get_substates(TARGET_QUBIT_LABEL)[-1])
    return {f"P{index}": float(value) for index, value in enumerate(np.real(populations))}


def repeat_zx90_sequence(
    sequence,
    *,
    repetitions: int = 20,
    control_initial: str = "0",
) -> pd.DataFrame:
    rows = []
    for repeats in range(repetitions + 1):
        if repeats == 0:
            rows.append(
                {
                    "control_initial": control_initial,
                    "repeats": 0,
                    "P0": 1.0,
                    "P1": 0.0,
                    "P2": 0.0,
                    "target_bloch": [0.0, 0.0, 1.0],
                }
            )
            continue
        repeated_sequence = sequence.repeated(repeats)
        result = simulator.mesolve(
            controls=repeated_sequence,
            initial_state={CONTROL_QUBIT_LABEL: control_initial, TARGET_QUBIT_LABEL: "0"},
            n_samples=2,
        )
        rows.append(
            {
                "control_initial": control_initial,
                "repeats": repeats,
                **target_qubit_population(result),
                "target_bloch": np.round(result.get_bloch_vectors(TARGET_QUBIT_LABEL)[-1], 4).tolist(),
            }
        )
    return pd.DataFrame(rows)


zx_repeat_df = repeat_zx90_sequence(zx90_sequence, repetitions=20, control_initial="0")
display(zx_repeat_df)

fig = qx.viz.make_plot_figure(
    x=zx_repeat_df["repeats"],
    y=zx_repeat_df["P1"],
    title=f"Repeat sequence : {TARGET_QUBIT_LABEL}",
    xlabel="Number of repetitions",
    ylabel="Normalized signal",
    ylim=[0, 1],
)
fig.show()


## Measure state distributions and build a 2-state classifier

The QUBEX experiment flow is:

1. `measure_state_distribution(targets, n_states=2)` calls `measure_state({target: "g"})` and `measure_state({target: "e"})`;
2. `build_classifier(...)` collects the `kerneled` IQ arrays from those prepared-state measurements;
3. `StateClassifierGMM.fit(data[target], phase=reference_phase)` fits the classifier and reports assignment fidelities.

Offline we mirror that method structure. The `"e"` preparation uses the calibrated HPI pulse twice, so preparation errors and leakage affect the synthetic IQ mixture before the GMM is fit.


In [ ]:
CLASSIFIER_TARGET = CONTROL_QUBIT_LABEL
CLASSIFIER_SHOTS = 10_000
CLASSIFIER_SEED = 7

rng = np.random.default_rng(CLASSIFIER_SEED)
readout_centers = {
    0: 0.10 + 0.02j,
    1: 0.86 + 0.34j,
    2: 1.14 - 0.20j,
}
readout_sigma = 0.16


def prepared_state_populations(state: str) -> dict[int, float]:
    if state == "g":
        return {0: 1.0, 1: 0.0, 2: 0.0}
    if state == "e":
        result = simulate_control_pulse(qx.PulseArray([hpi_pulse] * 2), n_samples=2)
        populations = result._get_population(result.get_substates(CLASSIFIER_TARGET)[-1])
        return {index: float(value) for index, value in enumerate(np.real(populations))}
    raise ValueError(f"unsupported prepared state: {state}")


def sample_readout_iq(populations: dict[int, float], *, n_shots: int) -> np.ndarray:
    states = np.array(sorted(populations))
    probabilities = np.array([populations[state] for state in states], dtype=float)
    probabilities = probabilities / probabilities.sum()
    sampled_states = rng.choice(states, size=n_shots, p=probabilities)
    centers = np.array([readout_centers[int(state)] for state in sampled_states])
    noise = readout_sigma * (rng.normal(size=n_shots) + 1j * rng.normal(size=n_shots))
    return centers + noise


def measure_state_offline(state_by_target: dict[str, str], *, n_shots: int = CLASSIFIER_SHOTS) -> dict[str, np.ndarray]:
    return {
        target: sample_readout_iq(prepared_state_populations(state), n_shots=n_shots)
        for target, state in state_by_target.items()
    }


def measure_state_distribution_offline(
    targets: list[str],
    *,
    n_states: int = 2,
    n_shots: int = CLASSIFIER_SHOTS,
) -> list[dict[str, np.ndarray]]:
    states = ["g", "e", "f"][:n_states]
    return [
        measure_state_offline(dict.fromkeys(targets, state), n_shots=n_shots)
        for state in states
    ]


def build_classifier_offline(
    targets: list[str],
    *,
    n_states: int = 2,
    n_shots: int = CLASSIFIER_SHOTS,
):
    distributions = measure_state_distribution_offline(targets, n_states=n_states, n_shots=n_shots)
    data = {
        target: {
            state: distributions[state][target]
            for state in range(n_states)
        }
        for target in targets
    }
    classifiers = {
        target: StateClassifierGMM.fit(data[target], phase=0.0)
        for target in targets
    }
    classified = {}
    fidelities = {}
    for target in targets:
        classified[target] = []
        for state in range(n_states):
            predicted = classifiers[target].predict(data[target][state])
            counts = np.bincount(predicted, minlength=classifiers[target].n_states)
            classified[target].append({label: int(counts[label]) for label in range(len(counts))})
        fidelities[target] = [
            classified[target][state][state] / sum(classified[target][state].values())
            for state in range(n_states)
        ]
    return {
        "data": data,
        "classifiers": classifiers,
        "classified": classified,
        "readout_fidelities": fidelities,
        "average_readout_fidelity": {target: float(np.mean(fidelities[target])) for target in targets},
    }


prepared_populations = {
    state: prepared_state_populations(state)
    for state in ("g", "e")
}
print("prepared populations used for classifier data:")
print(prepared_populations)

classifier_result = build_classifier_offline([CLASSIFIER_TARGET], n_states=2, n_shots=CLASSIFIER_SHOTS)
binary_classifier = classifier_result["classifiers"][CLASSIFIER_TARGET]
classifier_data = classifier_result["data"][CLASSIFIER_TARGET]
print("classifier centers:", binary_classifier.centers)
print("confusion matrix:")
print(binary_classifier.confusion_matrix)
print("readout fidelities:", classifier_result["readout_fidelities"][CLASSIFIER_TARGET])
print("average readout fidelity:", classifier_result["average_readout_fidelity"][CLASSIFIER_TARGET])


In [ ]:
classification_rows = []
for prepared_state, iq in classifier_data.items():
    predicted = binary_classifier.predict(iq)
    counts = np.bincount(predicted, minlength=binary_classifier.n_states)
    classification_rows.append(
        {
            "prepared_state": prepared_state,
            **{f"classified_{state}": int(counts[state]) for state in range(binary_classifier.n_states)},
            "assignment_fidelity": float(counts[prepared_state] / np.sum(counts)),
        }
    )
classification_df = pd.DataFrame(classification_rows)
display(classification_df)

for prepared_state, iq in classifier_data.items():
    predicted = binary_classifier.predict(iq)
    fig = qx.viz.make_classification_figure(
        target=CLASSIFIER_TARGET,
        data=iq,
        labels=predicted,
        centers=binary_classifier.centers,
        stddevs=binary_classifier.stddevs,
    )
    fig.update_layout(title=f"{CLASSIFIER_TARGET} prepared as |{prepared_state}>")
    fig.show()


## Next step

Only after this standalone DRAG HPI and ZX90 repeat check passes should we feed provider-generated schedules into qxsimulator. If a scheduling-comparison population plot fails before this baseline is healthy, the problem is simulator calibration rather than scheduling.
